In [1]:
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, confusion_matrix
from PIL import Image
from tqdm import tqdm

In [2]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')

        pt = torch.exp(-ce_loss)

        focal_loss = ((1-pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss
    
class VinDrMLODataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        
        # Mapeamento Binário (BI-RADS 1, 2, 3 = Benigno(0) | BI-RADS 4, 5 = Maligno(1))
        self.label_map = {
            'BI-RADS 1': 0, 
            'BI-RADS 2': 0, 
            'BI-RADS 3': 0, 
            'BI-RADS 4': 1, 
            'BI-RADS 5': 1
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Caminho do DICOM
        img_path = f"{self.root_dir}/{row['study_id']}/{row['image_id']}.dicom"
        
        # Leitura e Normalização do DICOM
        ds = pydicom.dcmread(img_path)
        pixel_array = ds.pixel_array.astype(float)
        
        # Normalização Min-Max para a imagem médica
        pixel_array = (pixel_array - np.min(pixel_array)) / (np.max(pixel_array) - np.min(pixel_array))
        pixel_array = (pixel_array * 255).astype(np.uint8)
        
        # Converte para imagem PIL em RGB (necessário para os pesos da ResNet)
        image = Image.fromarray(pixel_array).convert('RGB')
        
        lateralidade = row['laterality'] 
        
        # Espelha a mama direita para que todas fiquem orientadas como a esquerda
        if lateralidade == 'R':
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            
        # Pega a classe e converte para Binário
        label = self.label_map[row['breast_birads']]
        
        # Aplica Transformações (Tensor, Resize, Normalize...)
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [3]:
# --- Transformações (Atenção: NÃO há RandomHorizontalFlip aqui!) ---
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Carregamento e Preparação do CSV ---
csv_path = "../dataset/vindr-mammo/breast-level_annotations.csv"
images_dir = "../dataset/vindr-mammo/images"

df_completo = pd.read_csv(csv_path)

# Filtra apenas MLO (Verifique se no CSV chama 'view' ou 'view_position')
df_mlo = df_completo[df_completo['view_position'] == 'MLO'].copy()

# Remove possíveis linhas sem BI-RADS anotado, se houver
df_mlo = df_mlo.dropna(subset=['breast_birads'])

# --- Split sem Data Leakage (por study_id) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df_mlo, groups=df_mlo['study_id']))

df_train = df_mlo.iloc[train_idx]
df_val = df_mlo.iloc[val_idx]

print(f"Total Imagens MLO: {len(df_mlo)} | Treino: {len(df_train)} | Validação: {len(df_val)}")

# --- DataLoaders ---
BATCH_SIZE = 8

train_dataset = VinDrMLODataset(dataframe=df_train, root_dir=images_dir, transform=train_transform)
val_dataset = VinDrMLODataset(dataframe=df_val, root_dir=images_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

Total Imagens MLO: 9999 | Treino: 7999 | Validação: 2000


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Modelo ResNet50 ---
model = models.efficientnet_b4(weights='IMAGENET1K_V1')

# Troca a última camada para 2 classes (Benigno vs Maligno)
num_ftrs = model.classifier[1].in_features

model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_ftrs, 2)
)
model = model.to(device)

# --- Pesos das Classes para o Desbalanceamento ---
# Casos malignos são minoria. Damos um peso maior para a Classe 1 (Maligno)
# Exemplo: Peso 1.0 para Benigno e 10.0 para Maligno. (Ajuste se precisar de mais sensibilidade)
weights = torch.tensor([1.0, 10.0]).to(device) 

criterion = FocalLoss(weight=weights, gamma=2.0)

# Optimizer com um Learning Rate baixo, ideal para transfer learning
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

In [5]:
num_epochs = 50
best_auc = 0.0
TRESHOLD = 0.50

for epoch in range(num_epochs):
    print(f"\n--- Época {epoch+1}/{num_epochs} ---")
    
    # ==================================
    # TREINAMENTO
    # ==================================
    model.train()
    train_loss = 0.0
    
    loop_treino = tqdm(train_loader, desc="Treinamento", leave=False)
    
    for images, labels in loop_treino:
        images = images.to(device)
        labels = labels.to(device).long()  
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        loop_treino.set_postfix(loss=loss.item())
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # ==================================
    # VALIDAÇÃO
    # ==================================
    model.eval()
    val_loss = 0.0
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        loop_val = tqdm(val_loader, desc="Validação", leave=False)
        
        for images, labels in loop_val:
            images = images.to(device)
            labels = labels.to(device).long()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            
            probs = F.softmax(outputs, dim=1)[:, 1]
            
            preds = (probs >= TRESHOLD).long()
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    val_loss = val_loss / len(val_loader.dataset)
    
    # ==================================
    # MÉTRICAS
    # ==================================
    try:
        tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()
    except ValueError:
        # caso raro: só uma classe presente
        tn = fp = fn = tp = 0
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.0
    
    print(f"Loss Treino: {train_loss:.4f} | Loss Validação: {val_loss:.4f}")
    print(f"Matriz de Confusão -> TP:{tp} | FN:{fn} | TN:{tn} | FP:{fp}")
    print(f"Sensibilidade (Recall): {sensitivity:.4f}")
    print(f"Especificidade:       {specificity:.4f}")
    print(f"AUC-ROC:              {auc:.4f}")
    
    # ==================================
    # SALVAR MELHOR MODELO
    # ==================================
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'efficientnet_vindr_mlo_binario.pth')
        print(f"🔥 Novo melhor modelo salvo! (AUC: {best_auc:.4f})")


--- Época 1/50 ---


Loss Treino: 0.5118 | Loss Validação: 0.4575
Matriz de Confusão -> TP:16 | FN:73 | TN:1871 | FP:40
Sensibilidade (Recall): 0.1798
Especificidade:       0.9791
AUC-ROC:              0.6283
🔥 Novo melhor modelo salvo! (AUC: 0.6283)

--- Época 2/50 ---


Loss Treino: 0.4967 | Loss Validação: 0.4323
Matriz de Confusão -> TP:26 | FN:63 | TN:1831 | FP:80
Sensibilidade (Recall): 0.2921
Especificidade:       0.9581
AUC-ROC:              0.6578
🔥 Novo melhor modelo salvo! (AUC: 0.6578)

--- Época 3/50 ---


Loss Treino: 0.4841 | Loss Validação: 0.4159
Matriz de Confusão -> TP:32 | FN:57 | TN:1793 | FP:118
Sensibilidade (Recall): 0.3596
Especificidade:       0.9383
AUC-ROC:              0.7091
🔥 Novo melhor modelo salvo! (AUC: 0.7091)

--- Época 4/50 ---


Loss Treino: 0.4561 | Loss Validação: 0.3903
Matriz de Confusão -> TP:55 | FN:34 | TN:1387 | FP:524
Sensibilidade (Recall): 0.6180
Especificidade:       0.7258
AUC-ROC:              0.7318
🔥 Novo melhor modelo salvo! (AUC: 0.7318)

--- Época 5/50 ---


...

Loss Treino: 0.0332 | Loss Validação: 1.0023
Matriz de Confusão -> TP:40 | FN:49 | TN:1790 | FP:121
Sensibilidade (Recall): 0.4494
Especificidade:       0.9367
AUC-ROC:              0.7738

--- Época 47/50 ---


Loss Treino: 0.0374 | Loss Validação: 1.0284
Matriz de Confusão -> TP:40 | FN:49 | TN:1784 | FP:127
Sensibilidade (Recall): 0.4494
Especificidade:       0.9335
AUC-ROC:              0.7652

--- Época 48/50 ---


Loss Treino: 0.0308 | Loss Validação: 1.2477
Matriz de Confusão -> TP:33 | FN:56 | TN:1886 | FP:25
Sensibilidade (Recall): 0.3708
Especificidade:       0.9869
AUC-ROC:              0.7751

--- Época 49/50 ---


Loss Treino: 0.0220 | Loss Validação: 1.4008
Matriz de Confusão -> TP:35 | FN:54 | TN:1864 | FP:47
Sensibilidade (Recall): 0.3933
Especificidade:       0.9754
AUC-ROC:              0.7635

--- Época 50/50 ---


Loss Treino: 0.0317 | Loss Validação: 1.3467
Matriz de Confusão -> TP:37 | FN:52 | TN:1878 | FP:33
Sensibilidade (Recall): 0.4157
Especificidade:       0.9827
AUC-ROC:              0.7527
